&nbsp;

# ENCODE-rE2G Metadata Parse Notebook

- 以下のコマンドで収集した `bedpe`ファイルの JSONファイルをパースする

&nbsp;

```bash
curl -sS -H 'Accept: application/json' \ 
  'https://www.encodeproject.org/search/?type=Annotation&status=released&annotation_type=element+gene+regulatory+interaction+predictions&software_used.software.name=distal-regulation-encode_re2g&assembly=GRCh38&files.file_type=bedpe&frame=embedded&limit=all&format=json' \
  -o Data/ENCODE-rE2G_ctrl/metadata/encode_re2g_annotations_bedpe.json
```

&nbsp;

In [1]:
import polars as pl
from IPython.display import display

In [2]:
encode_re2g_annotations_bedpe = pl.read_json(
    "../../out/ENCODE_rE2G/ctrl_list/metadata/encode_re2g_annotations_bedpe.json"
).select(
    # Keep only the result list to avoid name clashes with page-level keys like "@id"
    "@graph"
).explode(
    "@graph", empty_as_null=True
).unnest(
    "@graph"
)

pl.Config.set_tbl_rows(1)
display(encode_re2g_annotations_bedpe)

shape: (1_460, 38)
┌───────────┬──────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬───────────┐
│ experimen ┆ status   ┆ date_crea ┆ alternate ┆ … ┆ aliases   ┆ dbxrefs   ┆ disease_t ┆ disease_t │
│ tal_input ┆ ---      ┆ ted       ┆ _accessio ┆   ┆ ---       ┆ ---       ┆ erm_name  ┆ erm_id    │
│ ---       ┆ str      ┆ ---       ┆ ns        ┆   ┆ list[str] ┆ list[null ┆ ---       ┆ ---       │
│ list[str] ┆          ┆ str       ┆ ---       ┆   ┆           ┆ ]         ┆ list[str] ┆ list[str] │
│           ┆          ┆           ┆ list[null ┆   ┆           ┆           ┆           ┆           │
│           ┆          ┆           ┆ ]         ┆   ┆           ┆           ┆           ┆           │
╞═══════════╪══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪═══════════╡
│ ["/experi ┆ released ┆ 2024-06-0 ┆ []        ┆ … ┆ ["jesse-e ┆ []        ┆ null      ┆ null      │
│ ments/ENC ┆          ┆ 6T04:28:3 ┆           ┆   ┆ ngreitz:a ┆           ┆           ┆           │
│ SR217OHA/ ┆          ┆ 2.641426+ ┆           ┆   ┆ bc_predic ┆           ┆           ┆           │
│ "]        ┆          ┆ 00:…      ┆           ┆   ┆ tio…      ┆           ┆           ┆           │
│ …         ┆ …        ┆ …         ┆ …         ┆ … ┆ …         ┆ …         ┆ …         ┆ …         │
└───────────┴──────────┴───────────┴───────────┴───┴───────────┴───────────┴───────────┴───────────┘

In [3]:
encode_re2g_bedpe_model_map = encode_re2g_annotations_bedpe.select(
    # Keep dataset-level columns with explicit names before expanding files
    pl.col("accession").alias("annotation_accession"),
    "files",
).explode(
    "files", empty_as_null=True
).unnest(
    # Prefix file-level fields as "files_*" to avoid clashes such as "accession" and "@id"
    "files", separator="_"
).filter(
    pl.col("files_file_format") == "bedpe"
).with_columns(
    # Label the model by the basename of the submitted file name
    pl.when(
        pl.col("files_submitted_file_name").str.split("/").list.last().str.starts_with("encode_e2g")
    ).then(pl.lit("ENCODE-rE2G")).otherwise(pl.lit("ABC")).alias("predicted_model")
).with_columns(
    ("https://www.encodeproject.org" + pl.col("files_href")).alias("download_url"),
).pipe(
    # Drop columns that are entirely null, or entirely empty lists for list columns
    lambda df: df.select(
        s.name for s in df
        if not (
            s.null_count() == df.height
            or (s.dtype == pl.List and (s.list.len().fill_null(0) == 0).all())
        )
    )
).select(
    "annotation_accession",
    "predicted_model",
    "files_annotation_type",
    "files_accession",
    "files_file_type",
    "files_derived_from",
    "files_assembly",
    "files_biosample_ontology",
    "files_assay_term_name",
    "files_date_created",
    "files_output_type",
    "files_content_md5sum",
    "files_md5sum"
).filter(
    pl.col("predicted_model") == "ENCODE-rE2G"
).with_columns(
    # Keep only accessions in derived_from, e.g. "/files/ENCFF622GRD/" -> "ENCFF622GRD"
    pl.col("files_derived_from").list.eval(
        pl.element().str.strip_prefix("/files/").str.strip_suffix("/")
    )
).with_columns(
    # Join every list[str] column into a comma-separated string
    pl.col(pl.List(pl.String)).list.join(", ")
).rename(
    {
        "annotation_accession": "ENCODE-rE2G accession"
    }
).with_columns(
    # Extract the ontology ID at the end of the path, e.g. "/biosample-types/tissue_UBERON_0002626/" -> "UBERON:0002626"
    pl.col("files_biosample_ontology")
    .str.extract(r"([A-Za-z]+_\d+)/$", 1)
    .str.replace("_", ":")
    .alias("Biosample term id")
)

pl.Config.set_fmt_str_lengths(300)
pl.Config.set_tbl_rows(2)
display(encode_re2g_bedpe_model_map)

ENCODE-rE2G accession,predicted_model,files_annotation_type,files_accession,files_file_type,files_derived_from,files_assembly,files_biosample_ontology,files_assay_term_name,files_date_created,files_output_type,files_content_md5sum,files_md5sum,Biosample term id
str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""ENCSR498RUW""","""ENCODE-rE2G""","""element gene regulatory interaction predictions""","""ENCFF168XSD""","""bedpe""","""ENCFF622GRD""","""GRCh38""","""/biosample-types/tissue_UBERON_0002626/""","""DNase-seq""","""2024-06-06T21:47:52.930849+00:00""","""thresholded links""","""0408a5fdb7f1cf2a44cf09716e5101ba""","""b90ba731eee39829682d665f43a465f9""","""UBERON:0002626"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""ENCSR997TVR""","""ENCODE-rE2G""","""element gene regulatory interaction predictions""","""ENCFF547ARY""","""bedpe""","""ENCFF461QVU""","""GRCh38""","""/biosample-types/cell_line_EFO_0002784/""","""DNase-seq, ChIP-seq, CAGE, HiC""","""2025-01-24T01:49:44.372281+00:00""","""thresholded links""","""0e4355f4bd057fc7aa2fc3a502d94ce4""","""bc1f0f0cf05fe8194cf9ea143f9d9031""","""EFO:0002784"""


&nbsp;

&nbsp;

&nbsp;

&nbsp;

## Metadata Intergration

- [An encyclopedia of human enhancer–gene regulatory interactions](https://doi.org/10.1038/s41586-026-10781-4) の Supplementary Table 12 と統合する

In [4]:
metadata_supplementary_table_12 = pl.read_csv(
    "../../out/ENCODE_rE2G/ctrl_list/metadata/ENCODE-rE2G_list_supplementary_table_12.tsv",
    separator="\t",
    n_threads=18
)

display(metadata_supplementary_table_12)

Model,ENCODE-rE2G accession,Sample,Biosample term name,Biosample term id,Biosample type,Biosample organism,Biosample treatments,Biosample treatments amount,Biosample treatments duration,Biosample genetic modifications methods,Biosample genetic modifications categories,Biosample genetic modifications targets,Biosample genetic modifications gene targets,Biosample genetic modifications site coordinates,Biosample genetic modifications zygosity,DNase Experiment accession,DNase Experiment date released,DNase Lab,DNase Donor(s),DNase File accession,DNase Output type,DNase Run type,ENCODE-rE2G v1,ENCODE Hi-C Experiment accession,H3K27ac Experiment accession,H3K4me3 Experiment accession,CAGE Experiment accession,EP300 Experiment accession,ChIA-PET Experiment accession,CTCF Experiment accession,in situ Hi-C Experiment accession
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,bool,str,str,str,str,str,str,str,str
"""Extended""","""ENCSR997TVR""","""GM12878_ENCSR000EMT""","""GM12878""","""EFO:0002784""","""cell line""","""Homo sapiens""",null,null,null,null,null,null,null,null,null,"""ENCSR000EMT""","""5/4/11""","""John Stamatoyannopoulos, UW""","""/human-donors/ENCDO000AAK/""","""ENCFF467CXY, ENCFF940NSD""","""unfiltered alignments, unfiltered alignments""","""single-ended, single-ended""",true,"""ENCSR916MFV""","""ENCSR000AKC""","""ENCSR057BWO""","""ENCSR000CKA""","""ENCSR000DZD""","""ENCSR184YZV""","""ENCSR000DZN""","""ENCSR410MDC"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""DNase-only""","""ENCSR738AUI""","""vagina_ENCSR437AYW""","""vagina""","""UBERON:0000996""","""tissue""","""Homo sapiens""",null,null,null,null,null,null,null,null,null,"""ENCSR437AYW""","""10/1/17""","""John Stamatoyannopoulos, UW""","""/human-donors/ENCDO793LXB/""","""ENCFF552WGU""","""alignments""","""paired-ended""",false,"""ENCSR492BKE""",null,null,null,null,null,null,null


In [5]:
encode_re2g_bedpe_model_map_join = encode_re2g_bedpe_model_map.join(
    metadata_supplementary_table_12,
    on=["ENCODE-rE2G accession"],
    how="inner",
    suffix="_table12"
).with_columns(
    # Flag rows whose ontology ID differs between ENCODE and Table 12
    (pl.col("Biosample term id") != pl.col("Biosample term id_table12")).alias("term_id_mismatch")
)

encode_re2g_bedpe_model_map_join.write_csv(
    "../../out/ENCODE_rE2G/ctrl_list/metadata/ENCODE-rE2G_bedpe_map_supplementary_table_12_join.tsv",
    separator="\t"
)

display(encode_re2g_bedpe_model_map_join)

ENCODE-rE2G accession,predicted_model,files_annotation_type,files_accession,files_file_type,files_derived_from,files_assembly,files_biosample_ontology,files_assay_term_name,files_date_created,files_output_type,files_content_md5sum,files_md5sum,Biosample term id,Model,Sample,Biosample term name,Biosample term id_table12,Biosample type,Biosample organism,Biosample treatments,Biosample treatments amount,Biosample treatments duration,Biosample genetic modifications methods,Biosample genetic modifications categories,Biosample genetic modifications targets,Biosample genetic modifications gene targets,Biosample genetic modifications site coordinates,Biosample genetic modifications zygosity,DNase Experiment accession,DNase Experiment date released,DNase Lab,DNase Donor(s),DNase File accession,DNase Output type,DNase Run type,ENCODE-rE2G v1,ENCODE Hi-C Experiment accession,H3K27ac Experiment accession,H3K4me3 Experiment accession,CAGE Experiment accession,EP300 Experiment accession,ChIA-PET Experiment accession,CTCF Experiment accession,in situ Hi-C Experiment accession,term_id_mismatch
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,bool,str,str,str,str,str,str,str,str,bool
"""ENCSR997TVR""","""ENCODE-rE2G""","""element gene regulatory interaction predictions""","""ENCFF547ARY""","""bedpe""","""ENCFF461QVU""","""GRCh38""","""/biosample-types/cell_line_EFO_0002784/""","""DNase-seq, ChIP-seq, CAGE, HiC""","""2025-01-24T01:49:44.372281+00:00""","""thresholded links""","""0e4355f4bd057fc7aa2fc3a502d94ce4""","""bc1f0f0cf05fe8194cf9ea143f9d9031""","""EFO:0002784""","""Extended""","""GM12878_ENCSR000EMT""","""GM12878""","""EFO:0002784""","""cell line""","""Homo sapiens""",null,null,null,null,null,null,null,null,null,"""ENCSR000EMT""","""5/4/11""","""John Stamatoyannopoulos, UW""","""/human-donors/ENCDO000AAK/""","""ENCFF467CXY, ENCFF940NSD""","""unfiltered alignments, unfiltered alignments""","""single-ended, single-ended""",true,"""ENCSR916MFV""","""ENCSR000AKC""","""ENCSR057BWO""","""ENCSR000CKA""","""ENCSR000DZD""","""ENCSR184YZV""","""ENCSR000DZN""","""ENCSR410MDC""",false
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""ENCSR738AUI""","""ENCODE-rE2G""","""element gene regulatory interaction predictions""","""ENCFF167ZIP""","""bedpe""","""ENCFF006OCW""","""GRCh38""","""/biosample-types/tissue_UBERON_0000996/""","""DNase-seq""","""2024-06-06T21:25:01.731850+00:00""","""thresholded links""","""571713665cbd75a5aea7f4ca07df12ab""","""7d0b9defe79d165dbc772dfb753eebf3""","""UBERON:0000996""","""DNase-only""","""vagina_ENCSR437AYW""","""vagina""","""UBERON:0000996""","""tissue""","""Homo sapiens""",null,null,null,null,null,null,null,null,null,"""ENCSR437AYW""","""10/1/17""","""John Stamatoyannopoulos, UW""","""/human-donors/ENCDO793LXB/""","""ENCFF552WGU""","""alignments""","""paired-ended""",false,"""ENCSR492BKE""",null,null,null,null,null,null,null,false


&nbsp;

&nbsp;

&nbsp;

&nbsp;

## `bed 3+` - `bedpe` Accession Map

- ダウンロードした `bed 3+` 形式のファイル（[`../../out/ENCODE_rE2G/ctrl_list/ENCODE-rE2G_ctrl_enhancer_gene_interaction`](../../out/ENCODE_rE2G/ctrl_list/ENCODE-rE2G_ctrl_enhancer_gene_interaction)）と、`bedpe` ファイル由来のキー `re2g_dataset`（例: `ENCSR000MZA_ENCFF654AMU`）の対応表を作成する
- `encode_re2g_annotations_bedpe` の `files` には、各 Annotation に含まれる全ファイルの情報が入っているため、Annotation のアクセッション（`ENCSR...`）を介して対応づけられる
- ENCODE-rE2G のファイルは `preferred_default == True` で ABC のファイルと区別できる
    - `bed 3+`: `output_type == "thresholded element gene links"`
    - `bedpe`: `output_type == "thresholded links"`
- 両ファイルが同じ予測結果から作られていることを `derived_from` の一致で確認する

&nbsp;

In [6]:
# Expand all files per annotation and keep only ENCODE-rE2G files (preferred_default == True)
encode_re2g_files = encode_re2g_annotations_bedpe.select(
    pl.col("accession").alias("annotation_accession"),
    "files",
).explode(
    "files", empty_as_null=True
).unnest(
    "files", separator="_"
).filter(
    pl.col("files_preferred_default") == True
).select(
    "annotation_accession",
    "files_accession",
    "files_file_type",
    "files_output_type",
    pl.col("files_derived_from").list.first().alias("files_derived_from"),
)

# Thresholded bed3+ files (the files downloaded into ENCODE-rE2G_ctrl_enhancer_gene_interaction)
encode_re2g_bed_map = encode_re2g_files.filter(
    (pl.col("files_file_type") == "bed bed3+")
    & (pl.col("files_output_type") == "thresholded element gene links")
).select(
    "annotation_accession",
    pl.col("files_accession").alias("bed_accession"),
    "files_derived_from",
)

# bedpe files (the files used to build re2g_dataset keys)
encode_re2g_bedpe_map = encode_re2g_files.filter(
    pl.col("files_file_type") == "bedpe"
).select(
    "annotation_accession",
    pl.col("files_accession").alias("bedpe_accession"),
    "files_derived_from",
)

# One bed3+ file per bedpe file; validate="1:1" raises an error if this ever breaks
encode_re2g_bed_bedpe_map = encode_re2g_bed_map.join(
    encode_re2g_bedpe_map,
    on="annotation_accession",
    how="inner",
    validate="1:1",
    suffix="_bedpe",
).with_columns(
    # Same key format as re2g_dataset in 04_encode_re2g_parse_ctrl.ipynb (e.g. ENCSR000MZA_ENCFF654AMU)
    (pl.col("annotation_accession") + "_" + pl.col("bedpe_accession")).alias("re2g_dataset"),
    # Both files should come from the same source predictions
    (pl.col("files_derived_from") == pl.col("files_derived_from_bedpe")).alias("derived_from_match"),
).select(
    "re2g_dataset",
    "annotation_accession",
    "bed_accession",
    "bedpe_accession",
    "derived_from_match",
).sort(
    "re2g_dataset"
)

print(f"rows: {encode_re2g_bed_bedpe_map.height}")
print(f"derived_from all match: {encode_re2g_bed_bedpe_map['derived_from_match'].all()}")

encode_re2g_bed_bedpe_map.write_csv(
    "../../out/ENCODE_rE2G/ctrl_list/metadata/encode_re2g_bed_bedpe_map.tsv",
    separator="\t"
)

pl.Config.set_tbl_rows(2)
display(encode_re2g_bed_bedpe_map)

rows: 1460
derived_from all match: True


re2g_dataset,annotation_accession,bed_accession,bedpe_accession,derived_from_match
str,str,str,str,bool
"""ENCSR000MZA_ENCFF654AMU""","""ENCSR000MZA""","""ENCFF727YIH""","""ENCFF654AMU""",true
…,…,…,…,…
"""ENCSR999YQC_ENCFF625FFH""","""ENCSR999YQC""","""ENCFF553ZXL""","""ENCFF625FFH""",true
